In [ ]:
!pip install pandas numpy nltk scikit-learn emoji tweepy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 9.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/dD2405/Twitter_Sentiment_Analysis/master/train.csv"
df = pd.read_csv(url)

df.head()

,id,label,tweet
0,1,0,@user when a father is dysfunctional and is s...
1,2,0,@user @user thanks for #lyft credit i can't us...
2,3,0,bihday your majesty
3,4,0,#model i love u take with u all the time in ...
4,5,0,factsguide: society now #motivation


Data Cleaning for Text Data
Steps:
Remove URLs
Remove mentions (@user)
Remove hashtags (#topic → keep word)
Remove emojis
Remove punctuation
Convert to lowercase

In [ ]:
import re
import emoji

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)  # remove URLs
    text = re.sub(r"@\w+", "", text)     # remove mentions
    text = re.sub(r"#", "", text)        # remove hashtag symbol
    text = emoji.replace_emoji(text, replace='')  # remove emojis
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # remove special chars
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['cleaned_tweet'] = df['tweet'].apply(clean_text)
df.head()

,id,label,tweet,cleaned_tweet
0,1,0,@user when a father is dysfunctional and is s...,when a father is dysfunctional and is so selfi...
1,2,0,@user @user thanks for #lyft credit i can't us...,thanks for lyft credit i cant use cause they d...
2,3,0,bihday your majesty,bihday your majesty
3,4,0,#model i love u take with u all the time in ...,model i love u take with u all the time in ur
4,5,0,factsguide: society now #motivation,factsguide society now motivation


String Manipulation Techniques (Preprocessing)

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')  # important fix

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    tokens = word_tokenize(text)  # Tokenization
    tokens = [w for w in tokens if w not in stop_words]  # Stopword removal
    tokens = [stemmer.stem(w) for w in tokens]  # Stemming
    return tokens

df['tokens'] = df['cleaned_tweet'].apply(preprocess)

df.head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


,id,label,tweet,cleaned_tweet,tokens
0,1,0,@user when a father is dysfunctional and is s...,when a father is dysfunctional and is so selfi...,"[father, dysfunct, selfish, drag, kid, dysfunc..."
1,2,0,@user @user thanks for #lyft credit i can't us...,thanks for lyft credit i cant use cause they d...,"[thank, lyft, credit, cant, use, caus, dont, o..."
2,3,0,bihday your majesty,bihday your majesty,"[bihday, majesti]"
3,4,0,#model i love u take with u all the time in ...,model i love u take with u all the time in ur,"[model, love, u, take, u, time, ur]"
4,5,0,factsguide: society now #motivation,factsguide society now motivation,"[factsguid, societi, motiv]"


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['processed_text'])
y = df['label']

print(X.shape)

(31962, 5000)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print("Accuracy:", accuracy)

Accuracy: 0.9540122008446739


In [ ]:
def tweet_stream(data):
    for tweet in data:
        yield tweet

stream = tweet_stream(df['cleaned_tweet'])

for i, tweet in enumerate(stream):
    if i == 5:
        break
    print(tweet)

when a father is dysfunctional and is so selfish he drags his kids into his dysfunction run
thanks for lyft credit i cant use cause they dont offer wheelchair vans in pdx disapointed getthanked
bihday your majesty
model i love u take with u all the time in ur
factsguide society now motivation


In [ ]:
batch_size = 1000

for i in range(0, len(df), batch_size):
    batch = df[i:i+batch_size]
    print(f"Processing batch {i} to {i+batch_size}")

Processing batch 0 to 1000
Processing batch 1000 to 2000
Processing batch 2000 to 3000
Processing batch 3000 to 4000
Processing batch 4000 to 5000
Processing batch 5000 to 6000
Processing batch 6000 to 7000
Processing batch 7000 to 8000
Processing batch 8000 to 9000
Processing batch 9000 to 10000
Processing batch 10000 to 11000
Processing batch 11000 to 12000
Processing batch 12000 to 13000
Processing batch 13000 to 14000
Processing batch 14000 to 15000
Processing batch 15000 to 16000
Processing batch 16000 to 17000
Processing batch 17000 to 18000
Processing batch 18000 to 19000
Processing batch 19000 to 20000
Processing batch 20000 to 21000
Processing batch 21000 to 22000
Processing batch 22000 to 23000
Processing batch 23000 to 24000
Processing batch 24000 to 25000
Processing batch 25000 to 26000
Processing batch 26000 to 27000
Processing batch 27000 to 28000
Processing batch 28000 to 29000
Processing batch 29000 to 30000
Processing batch 30000 to 31000
Processing batch 31000 to 3200

In [ ]:
print("""
Tools:
- Apache Kafka → Collect live tweets
- Spark Streaming → Process in real time
- Flink → Fast streaming analytics
""")


Tools:
- Apache Kafka → Collect live tweets
- Spark Streaming → Process in real time
- Flink → Fast streaming analytics

